# 🌱 Nostradamus IoTO API — Soil Data Example (SDK edition)

This notebook is the **SDK version** of the original `Nostradamus_IoTO_API_example_v1.ipynb`.
The original used raw `requests` calls against the REST API; here **every API interaction goes
through `nostradamus_ioto_sdk.NostradamusClient`**.

Workflow (same as the original):

1. Create a soil sensors collection with a schema
2. Generate and send soil sensor data (3 sensors × 24 readings)
3. Query data with limits, attribute selection, ordering, and filters
4. Compute statistics (avg / max / min / distinct)
5. Delete data for one sensor by key + timestamp range
6. Clean up the collection

## Prerequisites

```bash
pip install -e .          # from the repository root
pip install python-dotenv # optional, for loading a .env file
```

Environment variables:

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY`  | Write key (send data) |
| `NOSTRADAMUS_READ_KEY`   | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL`   | Optional API base URL override |


In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects the
`X-API-Key` header for you, so there is no manual header handling.


In [ ]:
import os
import random
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
        "NOSTRADAMUS_PROJECT_ID": PROJECT_ID,
        "NOSTRADAMUS_MASTER_KEY": MASTER_KEY,
        "NOSTRADAMUS_WRITE_KEY": WRITE_KEY,
        "NOSTRADAMUS_READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🌱 Nostradamus IoTO SDK — Soil Data Example")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the soil collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes straight
from the model. The demo is idempotent: if the service answers `409` because the collection
already exists, we list collections and reuse it.


In [ ]:
COLLECTION_NAME = "sensors"

SOIL_SCHEMA = {
    "key": "SOIL_001",                     # identifying key
    "timestamp": "2025-06-17T10:30:00Z",   # timestamp
    "soil_moisture": 45.2,                 # soil moisture %
    "ph_level": 6.8,                       # pH level
    "temperature": 18.5,                   # temperature °C
    "nitrogen": 45.0,                      # nitrogen ppm
    "phosphorus": 12.5,                    # phosphorus ppm
    "potassium": 180.0,                    # potassium ppm
    "battery_level": 87.3,                 # battery %
}

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Soil monitoring data",
            tags=["soil", "agriculture", "sensors"],
            collection_schema=SOIL_SCHEMA,
        )
        print("✅ Collection created successfully!")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send soil data

`data.send()` accepts a single dict or a list of dicts. The SDK serialises the payload and
authenticates the request with the write key.


In [ ]:
def generate_soil_data(num_sensors=3, num_readings=24):
    """Generate soil sensor readings for the last 24 hours."""
    data = []
    base_time = datetime.now(timezone.utc) - timedelta(hours=24)

    for sensor_num in range(1, num_sensors + 1):
        sensor_id = f"SOIL_{sensor_num:03d}"

        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)
            reading = {
                "key": sensor_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "soil_moisture": round(random.uniform(30, 60), 1),
                "ph_level": round(random.uniform(6.0, 7.5), 1),
                "temperature": round(random.uniform(15, 25), 1),
                "nitrogen": round(random.uniform(30, 60), 1),
                "phosphorus": round(random.uniform(8, 20), 1),
                "potassium": round(random.uniform(120, 220), 1),
                "battery_level": round(random.uniform(70, 100), 1),
            }
            data.append(reading)

    return data


soil_data = generate_soil_data(num_sensors=3, num_readings=24)

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=soil_data,
    )

print(f"✅ Sent {len(soil_data)} records")

## Step 3: Query data

`data.get()` returns a `DataListResponse` with `.data` (list of records) and
`.total_count`. Filters are passed as plain Python dictionaries — the SDK handles the JSON
encoding that the original notebook did by hand.


In [ ]:
def show(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    print(result.data)


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All data
    all_data = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    show(all_data, "📥 All data")

    # Limited data
    limit_data = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        limit=14,
    )
    show(limit_data, "📥 Limited data (14)")

    # Selected attributes, ordered by temperature descending
    ordered_data = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "temperature"],
        order_by='{"field": "temperature", "order": "desc"}',
    )
    show(ordered_data, "📥 key/temperature ordered desc")

### Filtering

The filter shape is unchanged (`property_name` / `operator` / `property_value`).
Multiple filter entries are combined as an AND.


In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # Specific sensor
    sensor_data = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "SOIL_001"}
        ],
    )
    show(sensor_data, "🔍 SOIL_001")

    # High moisture (> 45 %)
    high_moisture = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "soil_moisture", "operator": "gt", "property_value": 45}
        ],
    )
    show(high_moisture, "💧 soil_moisture > 45")

    # Low moisture AND low battery
    alert_data = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "soil_moisture", "operator": "lt", "property_value": 35},
            {"property_name": "battery_level", "operator": "lt", "property_value": 80},
        ],
    )
    show(alert_data, "⚠️ soil_moisture < 35 AND battery_level < 80")

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` / `interval` / `limit`.
Use the `StatOperation` enum instead of raw strings.

> **Note:** the SDK does not expose the `filters`/`order` parameters that the original
> notebook passed by hand. The equivalent of "min battery for SOIL_001" is the grouped
> aggregation `stat=MIN, group_by="key"` shown below.


In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    avg_moisture = read_client.data.statistics(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        stat=StatOperation.AVG,
        attribute="soil_moisture",
    )
    print(f"📈 avg soil_moisture: {avg_moisture}")

    max_temperature = read_client.data.statistics(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        stat=StatOperation.MAX,
        attribute="temperature",
    )
    print(f"📈 max temperature: {max_temperature}")

    min_battery_by_sensor = read_client.data.statistics(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        stat=StatOperation.MIN,
        attribute="battery_level",
        group_by="key",
    )
    print(f"📈 min battery_level by key: {min_battery_by_sensor}")

    distinct_keys = read_client.data.statistics(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        stat=StatOperation.DISTINCT,
        attribute="key",
    )
    print(f"📈 distinct keys: {distinct_keys}")

## Step 5: Delete data for one sensor

`data.delete()` accepts `key`, `timestamp_from`, `timestamp_to` — at least one is required.
It returns a `DeleteDataResponse` with `.message` and `.criteria`.


In [ ]:
def soil_001_window(read_client):
    return read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "timestamp"],
        order_by='{"field": "timestamp", "order": "asc"}',
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "SOIL_001"}
        ],
    )


timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    before = soil_001_window(master_client)
    show(before, f"📥 SOIL_001 before deletion ({timestamp_from} → {timestamp_to})")

    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key="SOIL_001",
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    print(f"🗑️  {result.message} (criteria={result.criteria})")

    after = soil_001_window(master_client)
    show(after, "📥 SOIL_001 after deletion")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.


In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )

print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")